### Preparación final de las Observaciones para K-Means (Pivotado)

**¿Qué estamos haciendo en este paso?**
Estamos transformando nuestro dataset limpio de un **formato "largo"** (donde cada fila es la medición de un solo contaminante a una hora específica) a un **formato "ancho" o matricial** (donde cada fila resume todo un día de mediciones en un punto específico).

Específicamente, el código realiza lo siguiente:

1. **Agrupa** los datos usando como identificadores únicos el `punto` de monitoreo y la `fecha`.


2. **Promedia** (`aggfunc='mean'`) las diferentes lecturas que un mismo contaminante tuvo durante ese día, obteniendo un valor representativo único.


3. **Pivota** la columna `contaminante`, convirtiendo los valores (PM10, PM2.5, SO2, NO2, CO) en **nuevas columnas independientes**.
4. **Filtra** (`dropna`) el dataset para conservar estrictamente las observaciones que cuenten con la información completa de los 5 contaminantes requeridos, descartando aquellas fechas/puntos con datos incompletos.



**¿Por qué es obligatorio hacer esto?**
Porque los algoritmos de Machine Learning, como K-Means, no pueden procesar filas sueltas de variables mezcladas. El algoritmo requiere que cada observación sea un **vector de características (Feature Vector)**.

En nuestro caso, de acuerdo con lo definido en la Sección 8.1 de nuestro informe, K-Means necesita calcular la distancia euclidiana basándose en vectores de 5 dimensiones: $X = [PM10, PM2.5, SO2, NO2, CO]$. Al aplicar este *script*, logramos estructurar matemáticamente los datos para pasárselos a nuestro código en el lenguaje Go, garantizando que los cálculos de distancias entre centroides sean precisos y correctos.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd

# Cargar el dataset limpio actual
ruta_actual = "/content/drive/MyDrive/Concurrente/Procesado/EAS_Aire_limpio.csv"
df = pd.read_csv(ruta_actual)

In [3]:
# Pivotar: Agrupar por punto y fecha, promediando las concentraciones
df_pivot = df.pivot_table(
    index=['punto', 'fecha'],
    columns='contaminante',
    values='concentracion',
    aggfunc='mean'
).reset_index()

In [4]:
# Filtrar observaciones completas
df_final = df_pivot.dropna(subset=['PM10', 'PM2.5', 'SO2', 'NO2', 'CO'])

In [5]:
# Guardar el dataset para Go
ruta_go = "/content/drive/MyDrive/Concurrente/Procesado/dataset_kmeans_go.csv"
df_final.to_csv(ruta_go, index=False)

print(f" Total de observaciones válidas: {len(df_final)}")
print(df_final.head())

 Total de observaciones válidas: 1035
contaminante      punto       fecha          CO       NO2       PM10  \
11587         CA-ILO-01  2022-01-01  319.170213  7.410000  23.200000   
11588         CA-ILO-01  2022-01-02  279.434783  6.584211  15.860000   
11589         CA-ILO-01  2022-01-03  277.976744  5.680000  12.713043   
11590         CA-ILO-01  2022-01-04  266.913043  8.622727  17.568000   
11591         CA-ILO-01  2022-01-05  266.940000  5.970000  17.220833   

contaminante      PM2.5        SO2  
11587         17.457143  21.399024  
11588          9.859091  69.512500  
11589          8.145455  36.874222  
11590         11.660870  22.319556  
11591          9.776190  22.355227  
